In [1]:

# ---- Shared runtime mode (full preserves this cell's original settings) ----
FAST_OPTION = globals().get("FAST_OPTION", "full")
if FAST_OPTION not in {"full", "balanced", "fast", "ultra"}:
    raise ValueError("FAST_OPTION must be one of: full, balanced, fast, ultra")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

# -------------------------------------------------------
# NUMPY 1.x / 2.x COMPATIBILITY
# -------------------------------------------------------
if hasattr(np, "trapezoid"):
    integrate_trapezoid = np.trapezoid
else:
    integrate_trapezoid = np.trapz

# -------------------------------------------------------
# CONFIG — aligned to the upgraded scripts
# -------------------------------------------------------
base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"
eeg_path = os.path.join(base_dir, "eeg_data_with_channels.npy")

out_dir = os.path.join(base_dir, "results", "exploratory_stft")
plots_dir = os.path.join(out_dir, "plots")
os.makedirs(out_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

sampling_rate = 1000
start_time, end_time = 814.571, 921.515
start_index, end_index = int(start_time * sampling_rate), int(end_time * sampling_rate)

# STFT is for time-frequency exploration, not stationary PSD truth.
# Downsample only for this pass so plotting and interpretation stay clean.
stft_downsample_factor = {"full": 4, "balanced": 6, "fast": 8, "ultra": 12}[FAST_OPTION]
fs_stft = sampling_rate / stft_downsample_factor  # 250 Hz

# STFT design
window_sec = 2.0
hop_sec = 0.5
nperseg = int(round(window_sec * fs_stft))        # 500 samples
noverlap = nperseg - int(round(hop_sec * fs_stft))# 375 samples
nfft = 1024
window_name = "hann"

# Frequency range of interest
fmin, fmax = 1.0, 100.0

# Preprocessing
apply_common_average_reference = False
detrend_mode = "constant"
zscore_each_channel = False

# Band definitions for envelope summaries
# Gamma excludes 58–62 Hz so line noise does not dominate the band envelope.
band_specs = {
    "delta":     [(1.0, 4.0)],
    "theta":     [(4.0, 8.0)],
    "alpha":     [(8.0, 13.0)],
    "beta":      [(13.0, 30.0)],
    "gamma":     [(30.0, 58.0), (62.0, 100.0)],
}
total_band_spec = [(1.0, 58.0), (62.0, 100.0)]
band_names = list(band_specs.keys())

# ROI definitions
roi_definitions = {
    "frontal":   ['Fp1', 'Fpz', 'Fp2', 'F3', 'Fz', 'F4'],
    "central":   ['C3', 'Cz', 'C4'],
    "posterior": ['P3', 'Pz', 'P4', 'O1', 'Oz', 'O2'],
}

eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

eps = 1e-12
ACCENT = "cyan"
BG = "black"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": ACCENT,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
    "font.size": 10,
})

# -------------------------------------------------------
# STYLE
# -------------------------------------------------------
def style_ax(ax, heatmap=False):
    ax.set_facecolor(BG)
    ax.tick_params(colors=ACCENT)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)
    if heatmap:
        ax.grid(False)
    else:
        ax.grid(True, alpha=0.18, color=ACCENT, linewidth=0.6)

# -------------------------------------------------------
# HELPERS
# -------------------------------------------------------
def ensure_time_channel_layout(x, n_channels_expected):
    x = np.asarray(x, dtype=float)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D EEG array, got shape {x.shape}")
    if x.shape[1] == n_channels_expected:
        return x
    if x.shape[0] == n_channels_expected:
        return x.T
    raise ValueError(f"Unexpected EEG shape {x.shape}; cannot identify channel axis.")

def maybe_common_average_reference(x, enabled=False):
    if not enabled:
        return x
    return x - np.mean(x, axis=1, keepdims=True)

def preprocess_channel(y, detrend_mode="constant", zscore=False):
    y = np.asarray(y, dtype=float)
    if detrend_mode is not None:
        y = signal.detrend(y, type=detrend_mode)
    if zscore:
        sd = np.std(y)
        y = (y - np.mean(y)) / (sd + eps)
    return y

def band_mask(freqs_hz, lo, hi):
    return (freqs_hz >= lo) & (freqs_hz <= hi)

def integrate_composite_band(freqs_hz, values, band_list, axis=0):
    total = None
    for lo, hi in band_list:
        mask = band_mask(freqs_hz, lo, hi)
        if np.sum(mask) < 2:
            continue
        piece = integrate_trapezoid(values[mask], freqs_hz[mask], axis=axis)
        if total is None:
            total = np.asarray(piece, dtype=float)
        else:
            total = total + np.asarray(piece, dtype=float)
    if total is None:
        if values.ndim == 1:
            return np.nan
        return np.full(values.shape[1], np.nan, dtype=float)
    return total

def area_normalize_1d(freqs_hz, x, band_list):
    x = np.asarray(x, dtype=float)
    out = np.full_like(x, np.nan, dtype=float)

    area = integrate_composite_band(freqs_hz, x, band_list, axis=0)
    if not np.isfinite(area) or area <= 0:
        return out

    mask = np.zeros_like(freqs_hz, dtype=bool)
    for lo, hi in band_list:
        mask |= band_mask(freqs_hz, lo, hi)

    out[mask] = x[mask] / area
    return out

def spectral_entropy(freqs_hz, spectrum_1d, band_list):
    mask = np.zeros_like(freqs_hz, dtype=bool)
    for lo, hi in band_list:
        mask |= band_mask(freqs_hz, lo, hi)
    mask &= np.isfinite(spectrum_1d)

    p = np.asarray(spectrum_1d[mask], dtype=float)
    p = np.maximum(p, 0.0)
    s = np.sum(p)
    if s <= 0:
        return np.nan
    p = p / s
    h = -np.sum(p * np.log(p + eps))
    hmax = np.log(len(p) + eps)
    return float(h / (hmax + eps))

def smooth_1d(x, win=7):
    x = np.asarray(x, dtype=float)
    if win <= 1 or len(x) < win:
        return x
    kernel = np.ones(win, dtype=float) / win
    return np.convolve(x, kernel, mode="same")

def smooth_time_series(x, fs, smoothing_sec=0.75):
    win = max(int(round(fs * smoothing_sec)), 1)
    if win % 2 == 0:
        win += 1
    return smooth_1d(x, win=win)

def zscore_1d(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    return (x - mu) / (sd + eps)

def robust_peak_in_band(freqs_hz, spectrum_1d, band=(7.0, 13.0), smooth_win=7):
    mask = band_mask(freqs_hz, band[0], band[1]) & np.isfinite(spectrum_1d)
    if np.sum(mask) < 5:
        return np.nan, np.nan, np.nan, np.nan

    f = freqs_hz[mask]
    y = smooth_1d(spectrum_1d[mask], win=smooth_win)

    peaks, _ = signal.find_peaks(y)
    if len(peaks) == 0:
        return np.nan, np.nan, np.nan, np.nan

    prominences = signal.peak_prominences(y, peaks)[0]
    valid = (peaks > 0) & (peaks < len(y) - 1)
    if not np.any(valid):
        return np.nan, np.nan, np.nan, np.nan

    peaks = peaks[valid]
    prominences = prominences[valid]

    min_prominence = max(0.05 * np.nanmax(y), 1.5 * np.nanmedian(np.abs(np.diff(y))))
    strong = prominences >= min_prominence
    if not np.any(strong):
        return np.nan, np.nan, np.nan, np.nan

    peaks = peaks[strong]
    prominences = prominences[strong]
    winner = int(np.argmax(prominences))
    peak_idx = peaks[winner]

    peak_freq = float(f[peak_idx])
    peak_height = float(y[peak_idx])
    peak_prominence = float(prominences[winner])
    peak_contrast = float(peak_height / (np.nanmedian(y) + eps))
    return peak_freq, peak_height, peak_prominence, peak_contrast

def percentile_limits(x, lo=2, hi=98):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return None, None
    return float(np.percentile(x, lo)), float(np.percentile(x, hi))

def roi_average_cube(cube, channel_names, roi_channels):
    idx = [channel_names.index(ch) for ch in roi_channels if ch in channel_names]
    if len(idx) == 0:
        return np.full(cube.shape[:2], np.nan, dtype=float)
    return np.nanmean(cube[:, :, idx], axis=2)

def roi_average_matrix(matrix, channel_names, roi_channels):
    idx = [channel_names.index(ch) for ch in roi_channels if ch in channel_names]
    if len(idx) == 0:
        return np.full(matrix.shape[0], np.nan, dtype=float)
    return np.nanmean(matrix[:, idx], axis=1)

def compute_stft_power(y, fs, nperseg, noverlap, nfft):
    """
    Expert correction relative to the original block:
    dB should be computed from power, not directly from |Zxx|.
    """
    freqs_hz, time_sec_rel, Zxx = signal.stft(
        y,
        fs=fs,
        window=window_name,
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        detrend=False,
        boundary=None,
        padded=False,
        return_onesided=True,
    )

    power = np.abs(Zxx) ** 2
    return freqs_hz, time_sec_rel, power

# -------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------
eeg_data = np.load(eeg_path, allow_pickle=True)
eeg_data = ensure_time_channel_layout(eeg_data, len(eeg_channel_names))

if end_index > eeg_data.shape[0]:
    raise ValueError(
        f"Requested end index {end_index} exceeds data length {eeg_data.shape[0]}. "
        "Check sampling rate or segment bounds."
    )

segment = eeg_data[start_index:end_index, :].copy()
segment = maybe_common_average_reference(segment, enabled=apply_common_average_reference)

segment_ds = signal.resample_poly(segment, up=1, down=stft_downsample_factor, axis=0)
segment_ds = np.asarray(segment_ds, dtype=float)

print(f"Loaded EEG data: {eeg_data.shape[0]} samples x {eeg_data.shape[1]} channels")
print(f"Segment of interest: {start_time:.3f} to {end_time:.3f} sec")
print(f"Original segment shape: {segment.shape}")
print(f"Downsampled STFT segment shape: {segment_ds.shape}")
print(f"STFT sampling rate: {fs_stft:.1f} Hz")

# -------------------------------------------------------
# CORE ANALYSIS
# -------------------------------------------------------
stft_data_dict = {}
summary_rows = []

freqs_hz_ref = None
time_sec_rel_ref = None

power_cube = None
log_power_cube = None
zfreq_cube = None

for ch_i, ch_name in enumerate(eeg_channel_names):
    y = preprocess_channel(
        segment_ds[:, ch_i],
        detrend_mode=detrend_mode,
        zscore=zscore_each_channel,
    )

    freqs_hz, time_sec_rel, power = compute_stft_power(
        y,
        fs=fs_stft,
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
    )

    keep = (freqs_hz >= fmin) & (freqs_hz <= fmax)
    freqs_hz = freqs_hz[keep]
    power = power[keep, :]

    log_power_db = 10.0 * np.log10(power + eps)
    zfreq = (log_power_db - np.nanmean(log_power_db, axis=1, keepdims=True)) / (
        np.nanstd(log_power_db, axis=1, keepdims=True) + eps
    )

    mean_spectrum = np.nanmean(power, axis=1)
    mean_spectrum_norm = area_normalize_1d(freqs_hz, mean_spectrum, total_band_spec)

    total_env = integrate_composite_band(freqs_hz, power, total_band_spec, axis=0)

    row = {
        "channel": ch_name,
        "spectral_entropy_1_100Hz": spectral_entropy(freqs_hz, mean_spectrum_norm, total_band_spec),
    }

    alpha_peak_hz, alpha_peak_height, alpha_peak_prominence, alpha_peak_contrast = robust_peak_in_band(
        freqs_hz, mean_spectrum_norm, band=(7.0, 13.0), smooth_win=7
    )
    row["alpha_peak_frequency_Hz"] = alpha_peak_hz
    row["alpha_peak_height_area_norm"] = alpha_peak_height
    row["alpha_peak_prominence"] = alpha_peak_prominence
    row["alpha_peak_contrast"] = alpha_peak_contrast
    row["has_clear_alpha_peak"] = bool(np.isfinite(alpha_peak_hz))

    band_payload = {}
    for band_name in band_names:
        abs_env = integrate_composite_band(freqs_hz, power, band_specs[band_name], axis=0)
        rel_env = abs_env / (total_env + eps)
        rel_env_logz = zscore_1d(np.log10(rel_env + eps))

        band_payload[f"{band_name}_power"] = abs_env
        band_payload[f"{band_name}_relative_power"] = rel_env
        band_payload[f"{band_name}_relative_power_logz"] = rel_env_logz

        row[f"{band_name}_power_mean"] = float(np.nanmean(abs_env))
        row[f"{band_name}_power_std"] = float(np.nanstd(abs_env))
        row[f"{band_name}_relative_mean"] = float(np.nanmean(rel_env))
        row[f"{band_name}_relative_std"] = float(np.nanstd(rel_env))
        row[f"{band_name}_relative_cv"] = float(np.nanstd(rel_env) / (np.nanmean(rel_env) + eps))
        row[f"{band_name}_burst_fraction_z_gt_1"] = float(np.nanmean(rel_env_logz > 1.0))
        row[f"{band_name}_burst_fraction_z_gt_2"] = float(np.nanmean(rel_env_logz > 2.0))

    alpha_rel = band_payload["alpha_relative_power"]
    beta_rel = band_payload["beta_relative_power"]
    gamma_rel = band_payload["gamma_relative_power"]

    row["alpha_beta_ratio_mean"] = float(np.nanmean(alpha_rel / (beta_rel + eps)))
    row["alpha_gamma_ratio_mean"] = float(np.nanmean(alpha_rel / (gamma_rel + eps)))
    row["alpha_gamma_corr"] = float(np.corrcoef(alpha_rel, gamma_rel)[0, 1]) if np.std(alpha_rel) > 0 and np.std(gamma_rel) > 0 else np.nan

    summary_rows.append(row)

    stft_data_dict[ch_name] = {
        "frequencies_hz": freqs_hz,
        "time_sec_rel": time_sec_rel,
        "time_sec_abs": start_time + time_sec_rel,
        "stft_log_power_db": log_power_db,
        "stft_zfreq": zfreq,
        **band_payload,
    }

    if freqs_hz_ref is None:
        freqs_hz_ref = freqs_hz
        time_sec_rel_ref = time_sec_rel
        n_freqs = len(freqs_hz_ref)
        n_times = len(time_sec_rel_ref)
        n_channels = len(eeg_channel_names)

        power_cube = np.full((n_freqs, n_times, n_channels), np.nan, dtype=float)
        log_power_cube = np.full((n_freqs, n_times, n_channels), np.nan, dtype=float)
        zfreq_cube = np.full((n_freqs, n_times, n_channels), np.nan, dtype=float)

    power_cube[:, :, ch_i] = power
    log_power_cube[:, :, ch_i] = log_power_db
    zfreq_cube[:, :, ch_i] = zfreq

summary_df = pd.DataFrame(summary_rows)
for col in summary_df.columns:
    if col != "channel":
        summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

time_sec_abs_ref = start_time + time_sec_rel_ref

# -------------------------------------------------------
# SAVE OUTPUTS
# -------------------------------------------------------
dict_npy_path = os.path.join(out_dir, "STFT_x.npy")
npz_path = os.path.join(out_dir, "exploratory_stft_outputs.npz")
summary_csv_path = os.path.join(out_dir, "stft_channel_summary.csv")
notes_txt_path = os.path.join(out_dir, "exploratory_stft_notes.txt")

np.save(dict_npy_path, stft_data_dict, allow_pickle=True)

np.savez_compressed(
    npz_path,
    frequencies_hz=freqs_hz_ref,
    time_sec_rel=time_sec_rel_ref,
    time_sec_abs=time_sec_abs_ref,
    power_cube=power_cube,
    log_power_cube=log_power_cube,
    zfreq_cube=zfreq_cube,
    channel_names=np.array(eeg_channel_names, dtype=object),
    band_names=np.array(band_names, dtype=object),
    start_time=start_time,
    end_time=end_time,
    original_sampling_rate=sampling_rate,
    stft_sampling_rate=fs_stft,
    stft_downsample_factor=stft_downsample_factor,
    nperseg=nperseg,
    noverlap=noverlap,
    nfft=nfft,
)

summary_df.to_csv(summary_csv_path, index=False)

top_alpha = summary_df.sort_values("alpha_relative_mean", ascending=False).head(8)
top_gamma = summary_df.sort_values("gamma_relative_mean", ascending=False).head(8)
top_alpha_bursty = summary_df.sort_values("alpha_burst_fraction_z_gt_1", ascending=False).head(8)

with open(notes_txt_path, "w") as f:
    f.write("Exploratory STFT notes\n")
    f.write("======================\n\n")
    f.write("Intent:\n")
    f.write("This pass is for exploratory time-frequency inspection of the selected EEG segment,\n")
    f.write("not a publication-ready pipeline, not a benchmark, and not a scoring system.\n\n")
    f.write("Key methodological upgrades relative to the original block:\n")
    f.write("1. The analysis is restricted to the selected recording interval.\n")
    f.write("2. STFT power is computed as |Zxx|^2 and converted to dB correctly.\n")
    f.write("3. The pass is downsampled only for this analysis so visualization stays interpretable and efficient.\n")
    f.write("4. Frequency-wise z-scored heatmaps are saved so transient structure is visible without one large line-noise bin flattening everything.\n")
    f.write("5. Absolute and relative band envelopes are saved alongside the STFT grids.\n")
    f.write("6. Gamma excludes 58–62 Hz in the band summaries so line noise does not dominate the envelope.\n\n")
    f.write(f"Segment: [{start_time:.3f}, {end_time:.3f}] sec\n")
    f.write(f"Original fs: {sampling_rate} Hz\n")
    f.write(f"STFT fs: {fs_stft:.1f} Hz\n")
    f.write(f"Downsample factor: {stft_downsample_factor}\n")
    f.write(f"Window: {window_sec:.2f} sec ({nperseg} samples)\n")
    f.write(f"Hop: {hop_sec:.2f} sec ({nperseg - noverlap} samples)\n")
    f.write(f"nfft: {nfft}\n")
    f.write(f"Frequency range analyzed: [{fmin:.1f}, {fmax:.1f}] Hz\n\n")

    f.write("Top alpha-relative-mean channels:\n")
    for _, row in top_alpha.iterrows():
        f.write(
            f"  {row['channel']}: "
            f"alpha_rel_mean={row['alpha_relative_mean']:.4f}, "
            f"alpha_peak={row['alpha_peak_frequency_Hz']}, "
            f"alpha_burst_fraction={row['alpha_burst_fraction_z_gt_1']:.3f}\n"
        )

    f.write("\nTop gamma-relative-mean channels:\n")
    for _, row in top_gamma.iterrows():
        f.write(
            f"  {row['channel']}: "
            f"gamma_rel_mean={row['gamma_relative_mean']:.4f}, "
            f"gamma_burst_fraction={row['gamma_burst_fraction_z_gt_1']:.3f}, "
            f"alpha_gamma_corr={row['alpha_gamma_corr']}\n"
        )

    f.write("\nTop alpha-bursty channels:\n")
    for _, row in top_alpha_bursty.iterrows():
        f.write(
            f"  {row['channel']}: "
            f"alpha_burst_fraction={row['alpha_burst_fraction_z_gt_1']:.3f}, "
            f"alpha_rel_mean={row['alpha_relative_mean']:.4f}\n"
        )

# -------------------------------------------------------
# PLOTS
# -------------------------------------------------------
alpha_idx = band_names.index("alpha")
gamma_idx = band_names.index("gamma")

# 1) Posterior ROI raw STFT log-power
posterior_power = roi_average_cube(power_cube, eeg_channel_names, roi_definitions["posterior"])
posterior_log_power = 10.0 * np.log10(posterior_power + eps)

fig, ax = plt.subplots(figsize=(13, 5.8), facecolor=BG)
style_ax(ax, heatmap=True)
mesh = ax.imshow(
    posterior_log_power,
    extent=[time_sec_abs_ref[0], time_sec_abs_ref[-1], freqs_hz_ref[0], freqs_hz_ref[-1]],
    aspect="auto",
    origin="lower",
    cmap="inferno",
)
ax.set_title("Posterior ROI STFT (log power)")
ax.set_xlabel("Time (sec)")
ax.set_ylabel("Frequency (Hz)")
ax.set_ylim(fmin, fmax)

cbar = plt.colorbar(mesh, ax=ax)
cbar.set_label("log10(power)", color=ACCENT)
cbar.outline.set_edgecolor(ACCENT)
cbar.ax.tick_params(color=ACCENT)
for label in cbar.ax.get_yticklabels():
    label.set_color(ACCENT)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "01_posterior_roi_stft_logpower.png"), dpi=180, bbox_inches="tight")
plt.show()
plt.close()

# 2) Posterior ROI frequency-wise z-scored STFT
posterior_zfreq = (posterior_log_power - np.nanmean(posterior_log_power, axis=1, keepdims=True)) / (
    np.nanstd(posterior_log_power, axis=1, keepdims=True) + eps
)
vmin, vmax = percentile_limits(posterior_zfreq, lo=2, hi=98)

fig, ax = plt.subplots(figsize=(13, 5.8), facecolor=BG)
style_ax(ax, heatmap=True)
mesh = ax.imshow(
    posterior_zfreq,
    extent=[time_sec_abs_ref[0], time_sec_abs_ref[-1], freqs_hz_ref[0], freqs_hz_ref[-1]],
    aspect="auto",
    origin="lower",
    cmap="coolwarm",
    vmin=vmin,
    vmax=vmax,
)
ax.set_title("Posterior ROI STFT (frequency-wise z score)")
ax.set_xlabel("Time (sec)")
ax.set_ylabel("Frequency (Hz)")
ax.set_ylim(fmin, fmax)

cbar = plt.colorbar(mesh, ax=ax)
cbar.set_label("z score within frequency", color=ACCENT)
cbar.outline.set_edgecolor(ACCENT)
cbar.ax.tick_params(color=ACCENT)
for label in cbar.ax.get_yticklabels():
    label.set_color(ACCENT)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "02_posterior_roi_stft_zfreq.png"), dpi=180, bbox_inches="tight")
plt.show()
plt.close()

# 3) Regional mean STFT spectra
region_spectra = {}
for region_name, roi_channels in roi_definitions.items():
    roi_power = roi_average_cube(power_cube, eeg_channel_names, roi_channels)
    mean_spectrum = np.nanmean(roi_power, axis=1)
    region_spectra[region_name] = area_normalize_1d(freqs_hz_ref, mean_spectrum, total_band_spec)

fig, ax = plt.subplots(figsize=(11.5, 5.5), facecolor=BG)
style_ax(ax)
ax.plot(freqs_hz_ref, region_spectra["frontal"], color="white", linewidth=1.8, alpha=0.95, label="Frontal ROI")
ax.plot(freqs_hz_ref, region_spectra["central"], color=ACCENT, linewidth=1.8, alpha=0.95, label="Central ROI")
ax.plot(freqs_hz_ref, region_spectra["posterior"], color="yellow", linewidth=2.0, alpha=0.95, label="Posterior ROI")
ax.axvspan(8.0, 13.0, color="white", alpha=0.05)
ax.set_xlim(fmin, fmax)
ax.set_title("Regional mean STFT spectra after area normalization")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Area-normalized mean STFT power")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "03_regional_mean_stft_spectra.png"), dpi=180, bbox_inches="tight")
plt.show()
plt.close()

# 4) Channel x frequency heatmap of mean STFT power
mean_spectrum_matrix = np.full((len(freqs_hz_ref), len(eeg_channel_names)), np.nan, dtype=float)
for ch_i, ch_name in enumerate(eeg_channel_names):
    mean_spectrum = np.nanmean(power_cube[:, :, ch_i], axis=1)
    mean_spectrum_matrix[:, ch_i] = area_normalize_1d(freqs_hz_ref, mean_spectrum, total_band_spec)

sort_df = summary_df.sort_values(
    by=["alpha_relative_mean", "alpha_peak_frequency_Hz"],
    ascending=[False, True],
    na_position="last",
).reset_index(drop=True)
sorted_channels = list(sort_df["channel"].values)
sorted_idx = [eeg_channel_names.index(ch) for ch in sorted_channels]

heat = np.log10(mean_spectrum_matrix[:, sorted_idx].T + 1e-6)
vmin, vmax = percentile_limits(heat, lo=2, hi=99)

fig, ax = plt.subplots(figsize=(12, 10), facecolor=BG)
style_ax(ax, heatmap=True)
im = ax.imshow(
    heat,
    aspect="auto",
    interpolation="nearest",
    cmap="inferno",
    vmin=vmin,
    vmax=vmax,
    extent=[freqs_hz_ref[0], freqs_hz_ref[-1], len(sorted_channels), 0],
)
ax.set_title("Channel × frequency heatmap of mean STFT power\n(sorted by alpha relative mean)")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Channel")
ax.set_yticks(np.arange(len(sorted_channels)) + 0.5)
ax.set_yticklabels(sorted_channels, fontsize=9)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("log10(area-normalized mean STFT power)", color=ACCENT)
cbar.outline.set_edgecolor(ACCENT)
cbar.ax.tick_params(color=ACCENT)
for label in cbar.ax.get_yticklabels():
    label.set_color(ACCENT)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "04_channel_frequency_heatmap_stft.png"), dpi=180, bbox_inches="tight")
plt.show()
plt.close()

# 5) Channel x time heatmap of alpha relative power
alpha_rel_matrix = np.full((len(time_sec_abs_ref), len(eeg_channel_names)), np.nan, dtype=float)
gamma_rel_matrix = np.full((len(time_sec_abs_ref), len(eeg_channel_names)), np.nan, dtype=float)

for ch_i, ch_name in enumerate(eeg_channel_names):
    alpha_rel_matrix[:, ch_i] = stft_data_dict[ch_name]["alpha_relative_power"]
    gamma_rel_matrix[:, ch_i] = stft_data_dict[ch_name]["gamma_relative_power"]

alpha_heat = np.full_like(alpha_rel_matrix.T, np.nan, dtype=float)
for i, ch_name in enumerate(sorted_channels):
    ch_idx = eeg_channel_names.index(ch_name)
    alpha_heat[i, :] = stft_data_dict[ch_name]["alpha_relative_power_logz"]

vmin, vmax = percentile_limits(alpha_heat, lo=2, hi=98)

fig, ax = plt.subplots(figsize=(13, 10), facecolor=BG)
style_ax(ax, heatmap=True)
im = ax.imshow(
    alpha_heat,
    aspect="auto",
    interpolation="nearest",
    cmap="coolwarm",
    vmin=vmin,
    vmax=vmax,
    extent=[time_sec_abs_ref[0], time_sec_abs_ref[-1], len(sorted_channels), 0],
)
ax.set_title("Channel × time heatmap of alpha relative power\n(sorted by alpha relative mean)")
ax.set_xlabel("Time (sec)")
ax.set_ylabel("Channel")
ax.set_yticks(np.arange(len(sorted_channels)) + 0.5)
ax.set_yticklabels(sorted_channels, fontsize=9)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("z score of log relative alpha power", color=ACCENT)
cbar.outline.set_edgecolor(ACCENT)
cbar.ax.tick_params(color=ACCENT)
for label in cbar.ax.get_yticklabels():
    label.set_color(ACCENT)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "05_channel_time_alpha_envelope_heatmap_stft.png"), dpi=180, bbox_inches="tight")
plt.show()
plt.close()

# 6) Regional alpha and gamma envelope dynamics
frontal_alpha = roi_average_matrix(alpha_rel_matrix, eeg_channel_names, roi_definitions["frontal"])
posterior_alpha = roi_average_matrix(alpha_rel_matrix, eeg_channel_names, roi_definitions["posterior"])
frontal_gamma = roi_average_matrix(gamma_rel_matrix, eeg_channel_names, roi_definitions["frontal"])
posterior_gamma = roi_average_matrix(gamma_rel_matrix, eeg_channel_names, roi_definitions["posterior"])

fig, ax = plt.subplots(figsize=(13, 5.2), facecolor=BG)
style_ax(ax)
ax.plot(time_sec_abs_ref, smooth_time_series(zscore_1d(np.log10(frontal_alpha + eps)), fs=1.0/hop_sec, smoothing_sec=1.0),
        color="white", linewidth=1.7, alpha=0.92, label="Frontal alpha")
ax.plot(time_sec_abs_ref, smooth_time_series(zscore_1d(np.log10(posterior_alpha + eps)), fs=1.0/hop_sec, smoothing_sec=1.0),
        color=ACCENT, linewidth=1.9, alpha=0.95, label="Posterior alpha")
ax.plot(time_sec_abs_ref, smooth_time_series(zscore_1d(np.log10(frontal_gamma + eps)), fs=1.0/hop_sec, smoothing_sec=1.0),
        color="yellow", linewidth=1.5, alpha=0.85, linestyle="--", label="Frontal gamma")
ax.plot(time_sec_abs_ref, smooth_time_series(zscore_1d(np.log10(posterior_gamma + eps)), fs=1.0/hop_sec, smoothing_sec=1.0),
        color="magenta", linewidth=1.5, alpha=0.85, linestyle="--", label="Posterior gamma")

ax.set_title("Regional alpha and gamma envelope dynamics from STFT")
ax.set_xlabel("Time (sec)")
ax.set_ylabel("z score of log relative band power")
ax.legend(loc="upper right", ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "06_regional_alpha_gamma_envelopes_stft.png"), dpi=180, bbox_inches="tight")
plt.show()
plt.close()

# 7) All-channel STFT grid using frequency-wise z scores
grid_vmin, grid_vmax = percentile_limits(zfreq_cube, lo=2, hi=98)

n_channels = len(eeg_channel_names)
n_cols = 4
n_rows = int(np.ceil(n_channels / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 2.8 * n_rows), facecolor=BG, sharex=True, sharey=True)
axes = np.asarray(axes).ravel()

for ch_i, ch_name in enumerate(eeg_channel_names):
    ax = axes[ch_i]
    style_ax(ax, heatmap=True)
    ax.imshow(
        zfreq_cube[:, :, ch_i],
        extent=[time_sec_abs_ref[0], time_sec_abs_ref[-1], freqs_hz_ref[0], freqs_hz_ref[-1]],
        aspect="auto",
        origin="lower",
        cmap="coolwarm",
        vmin=grid_vmin,
        vmax=grid_vmax,
    )
    ax.set_title(ch_name, fontsize=10)
    ax.set_ylim(fmin, fmax)

for ax in axes[n_channels:]:
    ax.axis("off")

for i, ax in enumerate(axes[:n_channels]):
    if i % n_cols == 0:
        ax.set_ylabel("Hz")
    if i >= (n_rows - 1) * n_cols:
        ax.set_xlabel("Time (sec)")

fig.suptitle("All-channel STFT heatmaps (frequency-wise z score)", color=ACCENT, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.savefig(os.path.join(plots_dir, "07_all_channel_stft_grid_zfreq.png"), dpi=180, bbox_inches="tight")
plt.show()
plt.close()

print(f"\nSaved dict NPY: {dict_npy_path}")
print(f"Saved NPZ: {npz_path}")
print(f"Saved summary CSV: {summary_csv_path}")
print(f"Saved notes TXT: {notes_txt_path}")
print(f"Saved plots to: {plots_dir}")
print("Done.")


Saved dict NPY: /home/a/projects/Complete-Neural-Signal-Analysis/results/exploratory_stft/STFT_x.npy
Saved NPZ: /home/a/projects/Complete-Neural-Signal-Analysis/results/exploratory_stft/exploratory_stft_outputs.npz
Saved summary CSV: /home/a/projects/Complete-Neural-Signal-Analysis/results/exploratory_stft/stft_channel_summary.csv
Saved notes TXT: /home/a/projects/Complete-Neural-Signal-Analysis/results/exploratory_stft/exploratory_stft_notes.txt
Saved plots to: /home/a/projects/Complete-Neural-Signal-Analysis/results/exploratory_stft/plots
Done.
